# Re-DocRED (revised) 학습 → 평가

`train_revised.json` 으로 학습하고 `dev_revised.json` 으로 매 epoch F1(dev_F1 / Ign_F1) 평가.

**실행 전 준비**: `data/docred_io.py`(revised split 추가)를 **dh 브랜치에 push** 해두세요.
위에서 아래로 순서대로 실행하면 됩니다. (A100 GPU 런타임 권장)


## 1) Google Drive 마운트

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) 코드(dh) + revised 데이터(main, LFS) 가져오기
- 코드는 `dh` 브랜치(최종 통합 모델)
- `train_revised`/`dev_revised` 는 `main`에 Git LFS로 있어 checkout 시 실체화됨


In [9]:
%cd /content
!apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install --skip-repo
!rm -rf project1                          # 이전 clone 있으면 제거 (안 그러면 clone 스킵→옛 코드 씀)
!git clone -q https://github.com/multiful/Information_Extraction.git project1
%cd /content/project1
!git checkout -q dh                       # 최종 모델 코드 + docred_io(revised split)
# revised 데이터는 main에 LFS로 있음 -> 가져와 실체화
!git checkout origin/main -- docred_data/data/train_revised.json docred_data/data/dev_revised.json
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json"
!pip install -q transformers==4.57.6 accelerate

# docred_io에 revised split 보장 (스테일 clone/미push 대비 런타임 패치)
import re, pathlib
_p = pathlib.Path("data/docred_io.py"); _s = _p.read_text(encoding="utf-8")
if "train_revised" not in _s:
    _p.write_text(re.sub(r'SPLITS\s*=\s*\[[^\]]*\]',
        'SPLITS = ["train_annotated","train_distant","dev","test","train_revised","dev_revised","test_revised"]',
        _s, count=1), encoding="utf-8")
    print("✅ docred_io SPLITS 런타임 패치 완료")
else:
    print("docred_io revised split 포함 OK")

import os
for f in ["train_revised", "dev_revised"]:
    kb = os.path.getsize(f"docred_data/data/{f}.json") // 1024
    print(f"{f}: {kb} KB", "OK" if kb > 100 else "  <-- ⚠️ LFS 실체화 실패(포인터). git lfs pull 확인")

/content
Git LFS initialized.
/content/project1
docred_io revised split 포함 OK
train_revised: 18166 KB OK
dev_revised: 3169 KB OK


## 3) 인코더(bert-base-cased) 로컬 다운로드
`from_pretrained`가 HF에서 받다가 멈추는 문제 우회: wget으로 직접 받아 로컬 폴더에 두고, 학습 때 `--model_name_or_path`로 그 폴더를 가리킴.


In [10]:
import os
os.makedirs("/content/bert-base-cased", exist_ok=True)
%cd /content/bert-base-cased
B  = "https://huggingface.co/google-bert/bert-base-cased/resolve/main"
UA = "Mozilla/5.0"
for f in ["config.json", "vocab.txt", "tokenizer.json", "tokenizer_config.json", "model.safetensors"]:
    print("↓", f, flush=True)
    !wget -c --tries=200 --timeout=30 --waitretry=3 --header="User-Agent: {UA}" -O {f} {B}/{f}
%cd /content/project1
!ls -lh /content/bert-base-cased/model.safetensors   # ~436M 이면 성공
# 계속 403/멈춤이면 위 B를 미러로: "https://hf-mirror.com/google-bert/bert-base-cased/resolve/main"

/content/bert-base-cased
↓ config.json
--2026-07-15 01:05:54--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 3.166.152.65, 3.166.152.44, 3.166.152.110, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.65|:443... connected.
HTTP request sent, awaiting response... 416 Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

↓ vocab.txt
--2026-07-15 01:05:54--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/vocab.txt
Resolving huggingface.co (huggingface.co)... 3.166.152.65, 3.166.152.44, 3.166.152.110, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.65|:443... connected.
HTTP request sent, awaiting response... 416 Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

↓ tokenizer.json
--2026-07-15 01:05:54--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/tokenizer.json
Resolving huggingface.co (huggingface.co

## 4) 학습(train_revised) + 평가(dev_revised)
- `--train_split train_revised --dev_split dev_revised` : revised로 학습/평가
- `--distant_mode none` : distant 사전학습 없이 train_revised 단독 학습
- 매 epoch 로그에 **dev_F1 / Ign_F1 / P / R** 출력


In [11]:
!python -u -m Scripts.atlop.train_full \
  --model_name_or_path /content/bert-base-cased \
  --train_split train_revised --dev_split dev_revised \
  --distant_mode none --epochs 20 --eval_batch_size 32 \
  --run_name atlop_full_revised --save_model

[device] cuda  [model] full (BERT node-GAT + DREAM evidence + gated MLP)
[data] train=3053 dev=500 docs
preprocess-full: 100% 500/500 [00:04<00:00, 118.05it/s]
preprocess-full: 100% 3053/3053 [00:22<00:00, 137.84it/s]
2026-07-15 01:06:34.625184: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-15 01:06:34.694745: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[stage 1] annotated train on 3053 docs (20 epoch(s))
  [annotated-train] epoch 0 step 50/764 loss 4.5720
  [annotated-train] epoch 0 step 100/764 loss 3.745

## 5) 결과 Drive 백업 (필수 — `/content`는 세션 종료 시 소실)
`dst` 를 본인 Drive 경로로 바꾸세요.


In [12]:
dst = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/atlop_full_revised.pt "{dst}/"
!cp results/atlop_full_revised_dev_predictions.json "{dst}/"
print("✅ Drive 저장 완료:", dst)

✅ Drive 저장 완료: /content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1
